# Multimodal House Price Prediction — Full Pipeline (v3, revision-ready)
**Dataset:** 52,852 properties (tabular + geo + amenity flags) + 262K listing image URLs.

**This version (v3)** is a direct response to peer-review feedback on v2. Every new section below is
tagged with the reviewer point(s) it addresses, so results map cleanly onto a response-to-reviewers
letter and revised manuscript sections.

| # | Reviewer concern | Where it's addressed |
|---|---|---|
| R1/R2 | n=3 seeds too small; no paired significance testing; concat beats x-attn on 2/3 seeds | `SEEDS` raised to 10 (Cell 2); new **paired significance testing** cell (Cell 10b) |
| R3 | Gated model's R² swings 0.39–0.70 across seeds; remedies (warm-up/annealing/entropy) proposed but never tested | New **gate-stability ablation** (Cell 10c): warm-up, temperature annealing, entropy regularization |
| R5 | Fusion operators compared under identical hyperparameters despite very different parameter counts (713K vs 366K) | New **`gated_wide`** parameter-matched control (Cell 7, Cell 10) |
| R6 | "Quality-aware" gate actually only measures photo *count*, not content quality | New genuine **blur-based sharpness signal** computed per image, fed into the gate alongside richness (Cell 4, Cell 8) |
| R7 | Single dataset / single backbone — generality unverified | `BACKBONE_NAME` swap hook + second-dataset config block (Cell 2, Cell 15) |
| R8 | Link survival (n_img) may be confounded with agency/price/locality — bias-ρ could be a proxy for something else | New **link-decay confound check** (Cell 5b): eta², Spearman, joint R² |
| minor | Bias figure doesn't show per-bin sample sizes | Bias figure now annotates `n` above each bar (Cell 13) |
| practical | Paper needs LaTeX-ready tables | New **LaTeX table exporter** (Cell 16) |

**Everything from v2 is preserved** — cell numbers below are inserted alongside the original flow, not a
rewrite from scratch, so you can diff against v2 cell-by-cell in your response letter.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 1 — Install dependencies (run once)
# !pip install torch timm pandas numpy scipy scikit-learn requests pillow matplotlib

In [ ]:
# Cell 2 — Imports & configuration (REVISED)
import os, io, csv, json, math
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from PIL import Image
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, wilcoxon, ttest_rel
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, TensorDataset

# ---------------- CONFIG ----------------
PROP_CSV   = "/content/drive/MyDrive/1 - Property_full_Table.csv"
PIC_CSV    = "/content/drive/MyDrive/4 - Picture.csv"
IMG_DIR    = "images"
MAX_IMG    = 10          # photos per property (dataset max)
WORKERS    = 16          # download threads
BATCH      = 512         # training batch size
EPOCHS     = 30
SEEDS      = list(range(10))   # REVISED (R1/R2): was [0,1,2]. 10 seeds so per-operator SD estimates
                                # and paired tests are defensible. Bump to 20 if compute allows --
                                # everything below is seed-count agnostic.
MOD_DROPOUT = 0.15       # modality dropout prob during training
HETEROSCEDASTIC = False  # True -> uncertainty-weighted variant
CHUNK      = 20_000      # images per embedding checkpoint
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# REVISED (R7): swap this string + rerun Cells 4-5 to get a second-backbone robustness check
# without touching anything downstream (embedding dim D is read from the model, not hardcoded).
#   primary run:   "vit_small_patch14_dinov2.lvd142m"   (DINOv2 ViT-S/14, self-supervised)
#   robustness run:"vit_base_patch16_clip_224.openai"    (CLIP ViT-B/16, matches MHPP's encoder family)
BACKBONE_NAME = "vit_small_patch14_dinov2.lvd142m"
BACKBONE_TAG  = BACKBONE_NAME.split(".")[0].replace("/", "_")   # used to namespace checkpoint files

# REVISED (R6): a real content-based quality signal (blur/sharpness) computed alongside the
# richness scalar, so the gate sees both "how many photos" and "how good are they" -- resolves
# the "quality-aware fusion only measures count" critique.
USE_QUALITY_SIGNAL = True
SHARP_RESIZE = 128       # downsample size for the Laplacian sharpness computation (cheap, fast)

# REVISED (R3): gate-stability remedies. GATE_STABILITY controls one experiment at a time;
# Cell 10c sweeps over all four settings and compares seed-to-seed R^2 variance.
#   "none"    -> v2 behaviour (unconditioned softmax gate)
#   "warmup"  -> gate temperature starts high (near-uniform gate) and anneals to 1.0 over
#                GATE_WARMUP_EPOCHS, so no pathway is starved of gradient early in training
#   "anneal"  -> same mechanism, slower/longer schedule across all EPOCHS
#   "entropy" -> adds an entropy bonus on the gate distribution z to the loss, discouraging
#                early collapse to a near-one-hot gate
GATE_STABILITY      = "none"
GATE_WARMUP_EPOCHS  = 5
GATE_TEMP_START     = 4.0
GATE_TEMP_END       = 1.0
GATE_ENTROPY_WEIGHT = 0.01

# REVISED (R5): parameter-matched control. "gated_wide" uses the same GMU as "gated" but with a
# wider regression head so its trainable-parameter count is close to "xattn"'s, isolating whether
# x-attn's edge comes from the cross-modal interaction mechanism or from extra capacity.
GATED_WIDE_HEAD_HIDDEN = 128   # auto-tuned in Cell 7b; this is just the starting guess

print("device:", DEVICE, "| backbone:", BACKBONE_NAME, "| seeds:", len(SEEDS))

# Checkpoint dir: Google Drive on Colab (survives disconnects), local otherwise
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = "/content/drive/MyDrive/house_price_ckpt"
except Exception:
    CKPT_DIR = "ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

## Step 1 — Download images (dead links ignored)
Unchanged from v2. Resume-safe: re-running skips files already on disk. Set `SAMPLE = 100` first for a
quick liveness probe before committing to all 262K URLs.

In [ ]:
# Cell 3 — Image downloader (unchanged from v2)
SAMPLE = 0   # 0 = everything; 100 = quick liveness probe first

HEADERS_HTTP = {"User-Agent": "Mozilla/5.0 (research; house-price-experiment)"}
MIN_BYTES = 5_000  # smaller = placeholder/error page -> dead

def fetch_one(row):
    pro_id, pic_no, url = row
    fname = f"{pro_id}_{pic_no}.jpg"
    fpath = os.path.join(IMG_DIR, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) >= MIN_BYTES:
        return (pro_id, pic_no, fname, "cached")
    for _ in range(2):
        try:
            r = requests.get(url, headers=HEADERS_HTTP, timeout=15)
            if r.status_code != 200 or len(r.content) < MIN_BYTES:
                continue
            Image.open(io.BytesIO(r.content)).verify()
            with open(fpath, "wb") as f:
                f.write(r.content)
            return (pro_id, pic_no, fname, "ok")
        except Exception:
            continue
    return (pro_id, pic_no, url, "dead")

os.makedirs(IMG_DIR, exist_ok=True)
pics = pd.read_csv(PIC_CSV)
if SAMPLE > 0:
    pics = pics.sample(SAMPLE, random_state=0)
rows = list(pics[["proID", "picNo", "picAddr"]].itertuples(index=False, name=None))

alive, n_dead = [], 0
with ThreadPoolExecutor(max_workers=WORKERS) as ex, \
     open("dead_links.csv", "w", newline="") as fdead:
    dead_writer = csv.writer(fdead); dead_writer.writerow(["proID", "picNo", "url"])
    futs = [ex.submit(fetch_one, r) for r in rows]
    for i, fut in enumerate(as_completed(futs)):
        pid, pno, name_or_url, status = fut.result()
        if status in ("ok", "cached"):
            alive.append((pid, pno, name_or_url))
        else:
            n_dead += 1; dead_writer.writerow([pid, pno, name_or_url])
        if (i + 1) % 5000 == 0:
            print(f"{i+1}/{len(rows)}  alive={len(alive)}  dead={n_dead}")

manifest = pd.DataFrame(alive, columns=["proID", "picNo", "filename"]).sort_values(["proID", "picNo"])
manifest.to_csv(os.path.join(CKPT_DIR, "image_manifest.csv"), index=False)
print(f"Alive: {len(manifest)}/{len(rows)} | properties with >=1 live image: "
      f"{manifest.proID.nunique()}/{pics.proID.nunique()}")

## Step 2 — Chunked embedding extraction + blur-sharpness signal (REVISED)
Same resume-safe chunking as v2, **plus** a per-image sharpness score (variance of the Laplacian on a
downsampled grayscale copy) computed in the same pass so it costs almost nothing extra. This is the
genuine content-quality signal requested by reviewer R6 — it captures blur/low-quality photos, which
photo *count* cannot.

To run the **second-backbone robustness check** (R7): change `BACKBONE_NAME` in Cell 2 and re-run this
cell — embeddings are namespaced by `BACKBONE_TAG` so you don't overwrite the primary run.

In [ ]:
# Cell 4 — Chunked embedding extraction with resume (REVISED: + sharpness signal, backbone-namespaced)
import timm

manifest = pd.read_csv(os.path.join(CKPT_DIR, "image_manifest.csv"))
backbone = timm.create_model(BACKBONE_NAME, pretrained=True, num_classes=0,
                             img_size=224).to(DEVICE).eval()
cfg = timm.data.resolve_data_config({}, model=backbone)
cfg["input_size"] = (3, 224, 224)      # FIX: transform must match resized model
transform = timm.data.create_transform(**cfg)
IN_SIZE = cfg["input_size"]

# sanity check before the long run
_x = transform(Image.new("RGB", (500, 400)))
assert tuple(_x.shape) == IN_SIZE, _x.shape
with torch.no_grad():
    assert backbone(_x[None].to(DEVICE)).shape[1] == backbone.num_features
print("shape sanity check OK:", _x.shape, "| backbone:", BACKBONE_NAME)

def sharpness_score(pil_img, size=SHARP_RESIZE):
    """Variance of Laplacian on a downsampled grayscale image -- a standard, cheap blur proxy.
    Low variance ~= blurry/flat image; high variance ~= sharp edges/detail."""
    g = np.asarray(pil_img.convert("L").resize((size, size)), dtype=np.float32)
    return float(ndimage.laplace(g).var())

class ImgDataset(Dataset):
    def __init__(self, mf): self.rows = mf.reset_index(drop=True)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        try:
            img = Image.open(os.path.join(IMG_DIR, r.filename)).convert("RGB")
            sharp = sharpness_score(img) if USE_QUALITY_SIGNAL else 0.0
            return transform(img), i, sharp
        except Exception:
            return torch.zeros(*IN_SIZE), -i - 1, 0.0   # FIX: matches transform shape

D = backbone.num_features
N = len(manifest)
emb_path   = os.path.join(CKPT_DIR, f"img_emb_{BACKBONE_TAG}.npy")
sharp_path = os.path.join(CKPT_DIR, f"img_sharp_{BACKBONE_TAG}.npy")
done_path  = os.path.join(CKPT_DIR, f"chunks_done_{BACKBONE_TAG}.json")
bad_path   = os.path.join(CKPT_DIR, f"bad_rows_{BACKBONE_TAG}.json")

if os.path.exists(emb_path):
    embs = np.lib.format.open_memmap(emb_path, mode="r+")
    assert embs.shape == (N, D), "manifest changed -- delete old checkpoint files"
else:
    embs = np.lib.format.open_memmap(emb_path, mode="w+", dtype=np.float32, shape=(N, D))

if os.path.exists(sharp_path):
    sharp_arr = np.lib.format.open_memmap(sharp_path, mode="r+")
    assert sharp_arr.shape == (N,)
else:
    sharp_arr = np.lib.format.open_memmap(sharp_path, mode="w+", dtype=np.float32, shape=(N,))

done = set(json.load(open(done_path))) if os.path.exists(done_path) else set()
bad  = set(json.load(open(bad_path)))  if os.path.exists(bad_path)  else set()

n_chunks = math.ceil(N / CHUNK)
print(f"{N} images, {n_chunks} chunks of {CHUNK}, {len(done)} already done")

for c in range(n_chunks):
    if c in done:
        continue
    lo, hi = c * CHUNK, min((c + 1) * CHUNK, N)
    dl = DataLoader(ImgDataset(manifest.iloc[lo:hi]), batch_size=128,
                    num_workers=2, pin_memory=True)   # FIX: 2 workers for Colab
    with torch.no_grad():
        for x, idx, sharp in dl:
            out = backbone(x.to(DEVICE)).cpu().numpy()
            sharp_np = sharp.numpy()
            for j, i_local in enumerate(idx.tolist()):
                if i_local < 0:
                    bad.add(lo + (-i_local - 1))
                else:
                    embs[lo + i_local] = out[j]
                    sharp_arr[lo + i_local] = sharp_np[j]
    embs.flush(); sharp_arr.flush()
    done.add(c)
    json.dump(sorted(done), open(done_path, "w"))
    json.dump(sorted(bad), open(bad_path, "w"))
    print(f"chunk {c+1}/{n_chunks} done (rows {lo}:{hi}, corrupt so far: {len(bad)})")

print(f"ALL DONE. embedded {N - len(bad)} images, {len(bad)} corrupt (masked), backbone={BACKBONE_NAME}")

In [ ]:
# Cell 5 — Build per-property padded tensors + masks (REVISED: + per-property mean sharpness)
manifest = pd.read_csv(os.path.join(CKPT_DIR, "image_manifest.csv"))
embs = np.load(os.path.join(CKPT_DIR, f"img_emb_{BACKBONE_TAG}.npy"), mmap_mode="r")
sharp_all = np.load(os.path.join(CKPT_DIR, f"img_sharp_{BACKBONE_TAG}.npy"), mmap_mode="r")
bad_path = os.path.join(CKPT_DIR, f"bad_rows_{BACKBONE_TAG}.json")
bad = set(json.load(open(bad_path))) if os.path.exists(bad_path) else set()
D = embs.shape[1]

props = pd.read_csv(PROP_CSV)
pro_ids = props["ID"].values
id_to_rows = manifest.reset_index().groupby("proID")["index"].apply(list).to_dict()

X = np.zeros((len(pro_ids), MAX_IMG, D), dtype=np.float32)
M = np.zeros((len(pro_ids), MAX_IMG), dtype=np.float32)
n_img = np.zeros(len(pro_ids), dtype=np.int32)
Q = np.zeros(len(pro_ids), dtype=np.float32)   # NEW: mean sharpness of live photos per property
for p, pid in enumerate(pro_ids):
    rws = [r for r in id_to_rows.get(pid, []) if r not in bad][:MAX_IMG]
    n_img[p] = len(rws)
    for k, r in enumerate(rws):
        X[p, k] = embs[r]; M[p, k] = 1.0
    if len(rws) > 0:
        Q[p] = float(np.mean([sharp_all[r] for r in rws]))

# guard: an all-zero embedding that slipped through counts as dead
zero_emb = (np.abs(X).sum(-1) == 0) & (M == 1)
M[zero_emb] = 0.0
n_img = M.sum(1).astype(np.int32)
Q[n_img == 0] = 0.0   # no live photos -> no quality signal (gate already sees richness=0)

# normalize sharpness to a roughly [0,1] range using train-safe later standardization (done in Cell 8);
# here we just log-compress since raw Laplacian variance is heavy-tailed
Q_log = np.log1p(Q)

np.savez_compressed(os.path.join(CKPT_DIR, f"property_images_{BACKBONE_TAG}.npz"),
                    emb=X, mask=M, n_img=n_img, quality=Q_log, proID=pro_ids)
zero = int((n_img == 0).sum())
print(f"saved to {CKPT_DIR}. properties with 0 live images: {zero} "
      f"({100*zero/len(pro_ids):.1f}%) -> mask handles them")
print(f"sharpness signal: mean={Q_log[n_img>0].mean():.3f}, std={Q_log[n_img>0].std():.3f} "
      f"(log1p-compressed variance-of-Laplacian, computed over live photos only)")

## Step 2b — Link-decay confound check (NEW — addresses R8)
Before building the model, check whether `n_img` (live-photo count) is associated with agency, locality,
postal code, price, or sale period. If it is, some of the measured "modality-imbalance bias" (residual vs.
photo count) could be a proxy for a different difficulty signal (e.g. budget agencies show fewer photos
*and* list harder-to-value stock) rather than photo count itself. This is a **diagnostic, not a fix** — it
tells you how much to hedge the RQ2 bias claim, and gives you exact numbers to cite in the limitations /
revised discussion section.

In [ ]:
# Cell 5b — Link-decay confound check (NEW, R8)
from sklearn.linear_model import Ridge

props_conf = pd.read_csv(PROP_CSV)
imgs_conf = np.load(os.path.join(CKPT_DIR, f"property_images_{BACKBONE_TAG}.npz"))
assert (props_conf["ID"].values == imgs_conf["proID"]).all()

conf_df = pd.DataFrame({
    "n_img": imgs_conf["n_img"].astype(float),
    "agency": props_conf["agency_name"].astype(str),
    "locality": props_conf["Locality"].astype(str),
    "postal": props_conf["Postal Code"].astype(str),
    "price": props_conf["price"].values.astype(float),
    "sold_date": props_conf["sold_date"].values,
})

def eta_squared(df, group_col, value_col="n_img"):
    """Proportion of variance in n_img explained by a categorical grouping variable."""
    grand_mean = df[value_col].mean()
    ss_between = df.groupby(group_col)[value_col].apply(
        lambda g: len(g) * (g.mean() - grand_mean) ** 2).sum()
    ss_total = ((df[value_col] - grand_mean) ** 2).sum()
    return float(ss_between / ss_total)

eta_agency   = eta_squared(conf_df, "agency")
eta_locality = eta_squared(conf_df, "locality")
eta_postal   = eta_squared(conf_df, "postal")
rho_price, p_price = spearmanr(conf_df["n_img"], conf_df["price"])
date_ord = pd.factorize(conf_df["sold_date"])[0]
rho_date, p_date = spearmanr(conf_df["n_img"], date_ord)

# Joint R^2: how well do agency + locality + price + sale period explain n_img together?
X_conf = pd.get_dummies(conf_df[["agency", "locality"]], drop_first=True)
X_conf["price"] = (conf_df["price"] - conf_df["price"].mean()) / conf_df["price"].std()
date_z = (date_ord - date_ord.mean()) / (date_ord.std() + 1e-6)
X_conf["date_ord"] = date_z
reg = Ridge(alpha=1.0).fit(X_conf, conf_df["n_img"])
r2_joint = float(reg.score(X_conf, conf_df["n_img"]))

print("=== Link-decay confound check: is n_img exogenous? ===")
print(f"eta^2   n_img ~ agency         : {eta_agency:.4f}")
print(f"eta^2   n_img ~ locality       : {eta_locality:.4f}")
print(f"eta^2   n_img ~ postal code    : {eta_postal:.4f}")
print(f"rho     n_img ~ price          : {rho_price:+.4f}  (p={p_price:.2e})")
print(f"rho     n_img ~ sale period    : {rho_date:+.4f}  (p={p_date:.2e})")
print(f"joint R^2 (agency+locality+price+period -> n_img): {r2_joint:.4f}")
print()
print("Interpretation: values well above 0 mean n_img is NOT exogenous -- report this joint R^2 "
      "explicitly next to the RQ2 bias-rho results, and hedge causal language accordingly "
      "(e.g. 'residual-count association, controlling partially for agency/locality/price').")

conf_summary = dict(eta_agency=eta_agency, eta_locality=eta_locality, eta_postal=eta_postal,
                     rho_price=float(rho_price), p_price=float(p_price),
                     rho_sale_period=float(rho_date), p_sale_period=float(p_date),
                     r2_joint=r2_joint)
json.dump(conf_summary, open(os.path.join(CKPT_DIR, "confound_check.json"), "w"), indent=2)

## Step 3 — Encoders (unchanged from v2)

In [ ]:
# Cell 6 — Encoders (unchanged from v2)
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(d_hidden, d_hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(d_hidden, d_out))
    def forward(self, x): return self.net(x)

class TabularEncoder(nn.Module):
    def __init__(self, n_numeric, cat_cardinalities, d_out=128):
        super().__init__()
        self.embs = nn.ModuleList(
            [nn.Embedding(c, min(32, (c + 1) // 2)) for c in cat_cardinalities])
        d_cat = sum(e.embedding_dim for e in self.embs)
        self.mlp = MLP(n_numeric + d_cat, 256, d_out)
    def forward(self, x_num, x_cat):
        cats = [e(x_cat[:, i]) for i, e in enumerate(self.embs)]
        return self.mlp(torch.cat([x_num] + cats, dim=1))

class ImageSetEncoder(nn.Module):
    """Attention pooling over up to MAX_IMG photos.
    Returns pooled vector + per-image attention weights (explainability).
    All-dead-images -> learned no-image token."""
    def __init__(self, d_img, d_out=128):
        super().__init__()
        self.proj = nn.Linear(d_img, d_out)
        self.attn = nn.Sequential(nn.Linear(d_out, d_out), nn.Tanh(), nn.Linear(d_out, 1))
        self.no_image = nn.Parameter(torch.zeros(d_out))
    def forward(self, emb, mask):
        h = self.proj(emb)
        a = self.attn(h).squeeze(-1).masked_fill(mask == 0, -1e9)
        w = torch.softmax(a, dim=1)
        pooled = torch.einsum("bk,bkd->bd", w, h)
        has_img = (mask.sum(1, keepdim=True) > 0).float()
        return has_img * pooled + (1 - has_img) * self.no_image, w

class GeoEncoder(nn.Module):
    """Random Fourier features over (lat,lng) + spatial-temporal comps feature."""
    def __init__(self, n_freq=32, d_out=64, d_extra=1):
        super().__init__()
        self.register_buffer("B", torch.randn(2, n_freq) * 10.0)
        self.mlp = MLP(2 * n_freq + d_extra, 128, d_out)
    def forward(self, latlng, extra):
        proj = latlng @ self.B
        return self.mlp(torch.cat([torch.sin(proj), torch.cos(proj), extra], dim=1))

## Step 4 — Fusion modules + FusionModel (REVISED)
Adds: (1) gate temperature + entropy hook for the stability ablation (R3), (2) a parameter-matched
`gated_wide` mode (R5).

In [ ]:
# Cell 7 — Fusion modules + FusionModel (REVISED: gate temperature/entropy hook, gated_wide mode)
class ModalityDropout(nn.Module):
    def __init__(self, p=0.15):
        super().__init__(); self.p = p
    def forward(self, feats):
        if not self.training or self.p == 0: return feats
        return [f * (torch.rand(f.size(0), 1, device=f.device) > self.p).float()
                for f in feats]

class GMU(nn.Module):
    """Gated Multimodal Unit; gate sees features + quality metadata.
    REVISED: forward() now accepts a temperature that divides the gate logits before softmax
    (temperature > 1 -> more uniform gate; used for warm-up/annealing, R3)."""
    def __init__(self, dims, d_out, d_meta):
        super().__init__()
        self.h = nn.ModuleList([nn.Sequential(nn.Linear(d, d_out), nn.Tanh()) for d in dims])
        self.gate = nn.Linear(sum(dims) + d_meta, len(dims))
    def forward(self, feats, meta, temperature=1.0):
        hs = torch.stack([h(f) for h, f in zip(self.h, feats)], dim=1)
        logits = self.gate(torch.cat(feats + [meta], dim=1)) / temperature
        z = torch.softmax(logits, dim=1)
        return torch.einsum("bm,bmd->bd", z, hs), z

class FusionModel(nn.Module):
    # REVISED (R5): added "gated_wide" -- identical GMU, wider head, parameter-matched to xattn.
    MODES = ("concat", "early", "late", "gated", "gated_wide", "xattn")
    def __init__(self, d_img_raw, n_numeric, cat_cards, d_text=0,
                 fusion="gated", d=128, heteroscedastic=False, modality_dropout=0.15,
                 gated_wide_head_hidden=128):
        super().__init__()
        assert fusion in self.MODES
        self.fusion, self.het, self.use_text = fusion, heteroscedastic, d_text > 0
        self.tab = TabularEncoder(n_numeric, cat_cards, d)
        self.img = ImageSetEncoder(d_img_raw, d)
        self.geo = GeoEncoder(d_out=d)
        if self.use_text: self.txt = nn.Linear(d_text, d)
        self.mdrop = ModalityDropout(modality_dropout)

        n_mod = 3 + int(self.use_text)
        d_meta, d_out_head = 2, (2 if heteroscedastic else 1)   # d_meta=2: [richness, quality] (R6)
        if fusion in ("concat", "early"):
            self.head = MLP(n_mod * d, 256, d_out_head)
        elif fusion == "late":
            self.heads = nn.ModuleList([MLP(d, 128, d_out_head) for _ in range(n_mod)])
            self.w = nn.Linear(n_mod * d + d_meta, n_mod)
        elif fusion == "gated":
            self.gmu = GMU([d] * n_mod, d, d_meta)
            self.head = MLP(d, 128, d_out_head)
        elif fusion == "gated_wide":
            self.gmu = GMU([d] * n_mod, d, d_meta)
            self.head = MLP(d, gated_wide_head_hidden, d_out_head)   # only diff vs "gated"
        elif fusion == "xattn":
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            self.mod_emb = nn.Parameter(torch.zeros(1, n_mod, d))
            layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=4 * d,
                                               batch_first=True, dropout=0.1)
            self.tf = nn.TransformerEncoder(layer, num_layers=2)
            self.head = MLP(d, 128, d_out_head)

    def forward(self, batch, gate_temperature=1.0):
        f_tab = self.tab(batch["x_num"], batch["x_cat"])
        f_img, img_attn = self.img(batch["img_emb"], batch["img_mask"])
        f_geo = self.geo(batch["latlng"], batch["comps"])
        feats = [f_tab, f_img, f_geo]
        if self.use_text: feats.append(self.txt(batch["txt_emb"]))
        feats = self.mdrop(feats)
        meta, aux = batch["mod_meta"], {"img_attn": img_attn}

        if self.fusion in ("concat", "early"):
            out = self.head(torch.cat(feats, dim=1))
        elif self.fusion == "late":
            preds = torch.stack([h(f) for h, f in zip(self.heads, feats)], 1)
            w = torch.softmax(self.w(torch.cat(feats + [meta], 1)), 1)
            aux["mod_weights"] = w
            out = torch.einsum("bm,bmo->bo", w, preds)
        elif self.fusion in ("gated", "gated_wide"):
            fused, z = self.gmu(feats, meta, temperature=gate_temperature)
            aux["mod_weights"] = z
            out = self.head(fused)
        else:  # xattn
            toks = torch.stack(feats, dim=1) + self.mod_emb
            toks = torch.cat([self.cls.expand(toks.size(0), -1, -1), toks], 1)
            out = self.head(self.tf(toks)[:, 0])

        if self.het:
            return out[:, 0], out[:, 1], aux
        return out.squeeze(1), None, aux

def gaussian_nll(mu, logvar, y):
    logvar = logvar.clamp(-10, 10)          # FIX: numerical stability
    return (0.5 * (logvar + (y - mu) ** 2 / logvar.exp())).mean()

### Cell 7b — Auto-tune `gated_wide` to match x-attn's parameter count (NEW, R5)
Binary-searches the head width so `gated_wide`'s trainable-parameter count lands within ~1% of
`xattn`'s, before any training happens. Run this once after Cell 8 (needs `META` to exist) and copy the
printed `GATED_WIDE_HEAD_HIDDEN` value back into Cell 2 if you want it persisted for future runs.

In [ ]:
# Cell 7b — Parameter-matching search for gated_wide vs xattn (NEW, R5). Run AFTER Cell 8.
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build(fusion, **kw):
    return FusionModel(META["d_img"], META["n_numeric"], META["cat_cards"],
                        d_text=META["d_text"], fusion=fusion,
                        heteroscedastic=HETEROSCEDASTIC, modality_dropout=MOD_DROPOUT, **kw)

target = count_params(build("xattn"))
lo, hi = 32, 1024
best_hidden, best_diff = GATED_WIDE_HEAD_HIDDEN, math.inf
for _ in range(12):
    mid = (lo + hi) // 2
    n = count_params(build("gated_wide", gated_wide_head_hidden=mid))
    if abs(n - target) < best_diff:
        best_diff, best_hidden = abs(n - target), mid
    if n < target:
        lo = mid + 1
    else:
        hi = mid - 1
    if lo > hi:
        break

GATED_WIDE_HEAD_HIDDEN = best_hidden
n_gated      = count_params(build("gated"))
n_gated_wide = count_params(build("gated_wide", gated_wide_head_hidden=GATED_WIDE_HEAD_HIDDEN))
n_xattn      = count_params(build("xattn"))
n_concat     = count_params(build("concat"))
n_late       = count_params(build("late"))
print(f"gated              : {n_gated:>8,} params")
print(f"gated_wide (h={GATED_WIDE_HEAD_HIDDEN:<4}) : {n_gated_wide:>8,} params  <- matched to xattn")
print(f"xattn              : {n_xattn:>8,} params  (target)")
print(f"concat              : {n_concat:>8,} params")
print(f"late               : {n_late:>8,} params")
print(f"\\n(gated_wide - xattn) = {n_gated_wide - n_xattn:+,} params "
      f"({100*(n_gated_wide-n_xattn)/n_xattn:+.2f}%)")

## Step 5 — Feature build + temporal split (REVISED: quality signal joins `mod_meta`)
`mod_meta` is now `[richness, quality]` instead of `[richness, text_len]` by default (text still supported
via Cell 13 if you extract description features — just append a 3rd column and update `d_meta` in Cell 7).

In [ ]:
# Cell 8 — Spatial-temporal comps + feature build + temporal split (REVISED: quality in mod_meta)
CAT_COLS = ["proType", "agency_name", "Locality", "Postal Code"]
DROP = ["ID", "price", "sold_date", "Lat", "Lng"] + CAT_COLS

def temporal_comps(df, k=10):
    """Median log-price of k nearest spatial neighbours sold strictly earlier."""
    df = df.sort_values("sold_date").reset_index()
    comps = np.full(len(df), np.nan)
    coords = np.nan_to_num(df[["Lat","Lng"]].values, nan=0.5)
    logp, dates = np.log(df["price"].values), df["sold_date"].values
    for t in np.unique(dates):
        prior = dates < t
        cur = np.where(dates == t)[0]
        if prior.sum() < k: continue
        tree = cKDTree(coords[prior])
        _, idx = tree.query(coords[cur], k=k)
        comps[cur] = np.median(logp[np.where(prior)[0][idx]], axis=1)
    return pd.Series(comps, index=df["index"]).sort_index().values

df = pd.read_csv(PROP_CSV)
imgs = np.load(os.path.join(CKPT_DIR, f"property_images_{BACKBONE_TAG}.npz"))
assert (df["ID"].values == imgs["proID"]).all(), "row order mismatch"

y_all = np.log(df["price"].values.astype(np.float32))
comps = temporal_comps(df).astype(np.float32)
num_cols = [c for c in df.columns if c not in DROP]
x_num = df[num_cols].values.astype(np.float32)
x_cat = np.stack([pd.factorize(df[c])[0] for c in CAT_COLS], 1).astype(np.int64)
x_cat[x_cat < 0] = 0                        # FIX: factorize gives -1 for NaN categories
cat_cards = [int(x_cat[:, i].max()) + 1 for i in range(x_cat.shape[1])]
latlng = np.nan_to_num(df[["Lat", "Lng"]].values.astype(np.float32), nan=0.5)

n_img_arr = imgs["n_img"].astype(np.float32)
quality_arr = imgs["quality"].astype(np.float32) if USE_QUALITY_SIGNAL else np.zeros(len(df), np.float32)
# REVISED (R6): mod_meta is now [richness, quality] -- both raw, standardized below with train stats
mod_meta = np.stack([n_img_arr / MAX_IMG, quality_arr], 1)

# temporal split
order = np.argsort(df["sold_date"].values, kind="stable")
n = len(df)
tr, va, te = order[:int(.7*n)], order[int(.7*n):int(.8*n)], order[int(.8*n):]

# FIX: impute NaN with TRAIN median (leak-free), then standardize + clip
x_num = np.where(np.isinf(x_num), np.nan, x_num)
train_med = np.nanmedian(x_num[tr], axis=0)
train_med = np.nan_to_num(train_med, nan=0.0)   # column entirely NaN in train
nan_mask = np.isnan(x_num)
x_num[nan_mask] = np.take(train_med, np.where(nan_mask)[1])
mu_, sd_ = x_num[tr].mean(0), x_num[tr].std(0) + 1e-6
x_num = np.clip((x_num - mu_) / sd_, -10, 10)

comps = np.nan_to_num(comps, nan=np.nanmedian(comps[tr]))
comps = (comps - comps[tr].mean()) / (comps[tr].std() + 1e-6)

# standardize the quality column with train stats only (richness column already in [0,1])
q_mu, q_sd = mod_meta[tr, 1].mean(), mod_meta[tr, 1].std() + 1e-6
mod_meta[:, 1] = (mod_meta[:, 1] - q_mu) / q_sd

TENSORS = dict(
    x_num=torch.tensor(x_num), x_cat=torch.tensor(x_cat),
    latlng=torch.tensor(latlng), comps=torch.tensor(comps[:, None]),
    img_emb=torch.tensor(imgs["emb"]), img_mask=torch.tensor(imgs["mask"]),
    mod_meta=torch.tensor(mod_meta), y=torch.tensor(y_all))
META = dict(cat_cards=cat_cards, d_img=imgs["emb"].shape[2],
            n_numeric=x_num.shape[1], n_img=n_img_arr, d_text=0)

# FIX: hard guarantee -- nothing NaN/Inf enters training
for k, v in TENSORS.items():
    if v.dtype.is_floating_point:
        assert not torch.isnan(v).any(), f"NaN in {k}"
        assert not torch.isinf(v).any(), f"Inf in {k}"
print(f"train {len(tr)} | val {len(va)} | test {len(te)} | "
      f"numeric feats {x_num.shape[1]} | mod_meta dims: [richness, quality] | all tensors NaN/Inf-free")

## Step 6 — Train/eval helpers (REVISED: gate temperature schedule + entropy regularization)

In [ ]:
# Cell 9 — Train/eval helpers (REVISED: gate temperature schedule, entropy regularization, R3)
def make_loader(tensors, idx, shuffle):
    keys = list(tensors.keys())
    ds = TensorDataset(*[tensors[k][idx] for k in keys])
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle), keys

def gate_temperature_for_epoch(ep, stability):
    """REVISED (R3): temperature schedule for gate warm-up / annealing.
    temperature=1.0 recovers exact v2 behaviour (GATE_STABILITY='none')."""
    if stability == "warmup":
        t = min(ep / max(GATE_WARMUP_EPOCHS, 1), 1.0)
        return GATE_TEMP_START + t * (GATE_TEMP_END - GATE_TEMP_START)
    if stability == "anneal":
        t = min(ep / max(EPOCHS - 1, 1), 1.0)
        return GATE_TEMP_START + t * (GATE_TEMP_END - GATE_TEMP_START)
    return 1.0

def run_epoch(model, dl, keys, opt=None, gate_temperature=1.0, entropy_weight=0.0):
    model.train(opt is not None)
    tot, cnt, preds, ys = 0.0, 0, [], []
    for batch in dl:
        b = {k: v.to(DEVICE) for k, v in zip(keys, batch)}
        y = b.pop("y")
        mu, logvar, aux = model(b, gate_temperature=gate_temperature)
        loss = gaussian_nll(mu, logvar, y) if logvar is not None \
            else torch.nn.functional.huber_loss(mu, y)
        # REVISED (R3): entropy regularization -- encourages a less peaked gate early in training
        if entropy_weight > 0 and "mod_weights" in aux:
            z = aux["mod_weights"].clamp_min(1e-8)
            entropy = -(z * z.log()).sum(dim=1).mean()
            loss = loss - entropy_weight * entropy
        if opt:
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # FIX
            opt.step()
        tot += loss.item() * len(y); cnt += len(y)
        preds.append(mu.detach().cpu()); ys.append(y.cpu())
    return tot / cnt, torch.cat(preds).numpy(), torch.cat(ys).numpy()

def metrics(pred_log, y_log):
    p, yv = np.exp(pred_log), np.exp(y_log)
    return dict(
        rmse_log=float(np.sqrt(np.mean((pred_log - y_log) ** 2))),
        mae_log=float(np.mean(np.abs(pred_log - y_log))),
        mape=float(np.mean(np.abs(p - yv) / yv) * 100),
        r2=float(1 - np.sum((p - yv) ** 2) / np.sum((yv - yv.mean()) ** 2)))

def train_one(fusion, seed, verbose=True, gate_stability=None, gated_wide_head_hidden=None):
    """REVISED: gate_stability overrides the global GATE_STABILITY per-call (used by the
    stability-ablation cell, R3); gated_wide_head_hidden overrides the global default (R5)."""
    stability = gate_stability if gate_stability is not None else GATE_STABILITY
    ghh = gated_wide_head_hidden if gated_wide_head_hidden is not None else GATED_WIDE_HEAD_HIDDEN
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModel(META["d_img"], META["n_numeric"], META["cat_cards"],
                        d_text=META["d_text"], fusion=fusion,
                        heteroscedastic=HETEROSCEDASTIC,
                        modality_dropout=MOD_DROPOUT,
                        gated_wide_head_hidden=ghh).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    dl_tr, keys = make_loader(TENSORS, tr, True)
    dl_va, _ = make_loader(TENSORS, va, False)
    dl_te, _ = make_loader(TENSORS, te, False)

    is_gated = fusion in ("gated", "gated_wide")
    ent_w = GATE_ENTROPY_WEIGHT if (is_gated and stability == "entropy") else 0.0

    best = np.inf
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # FIX: never None
    patience = 0
    for ep in range(EPOCHS):
        temp = gate_temperature_for_epoch(ep, stability) if is_gated else 1.0
        tr_loss, _, _ = run_epoch(model, dl_tr, keys, opt, gate_temperature=temp, entropy_weight=ent_w)
        if np.isnan(tr_loss):                                          # FIX: fail loudly
            raise RuntimeError(f"NaN loss in {fusion} -- inspect TENSORS")
        va_loss, _, _ = run_epoch(model, dl_va, keys, gate_temperature=1.0)  # eval always at temp=1
        if va_loss < best - 1e-4:
            best = va_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= 5: break
        if verbose:
            print(f"  [{fusion} s{seed} stab={stability}] ep{ep:02d} train {tr_loss:.4f} "
                  f"val {va_loss:.4f} temp={temp:.2f}")
    model.load_state_dict(best_state)

    _, pred, yv = run_epoch(model, dl_te, keys, gate_temperature=1.0)
    res = metrics(pred, yv)

    # ablation: images zeroed at test time
    t2 = {k: (torch.zeros_like(v) if k == "img_mask" else v) for k, v in TENSORS.items()}
    dl_ab, _ = make_loader(t2, te, False)
    _, pred_ab, _ = run_epoch(model, dl_ab, keys, gate_temperature=1.0)
    res["rmse_log_no_images"] = metrics(pred_ab, yv)["rmse_log"]
    res["delta_no_images"] = res["rmse_log_no_images"] - res["rmse_log"]

    # bias diagnosis: |residual| vs image count
    rho, pval = spearmanr(np.abs(pred - yv), META["n_img"][te])
    res["spearman_absresid_vs_nimg"] = float(rho)
    res["spearman_pval"] = float(pval)
    return res, model

## Step 7 — Run the full comparison (REVISED: 10 seeds, includes `gated_wide`)

In [ ]:
# Cell 10 — Run the full comparison (all fusions x all seeds) (REVISED: 10 seeds, gated_wide included)
RESULTS = {}
for fusion in FusionModel.MODES:
    per_seed = []
    for seed in SEEDS:
        r, _m = train_one(fusion, seed, verbose=False)
        per_seed.append(r)
    agg = {k: (float(np.mean([r[k] for r in per_seed])),
               float(np.std([r[k] for r in per_seed], ddof=1))) for k in per_seed[0]}
    RESULTS[fusion] = {"per_seed": per_seed, "mean_std": agg}
    print(f"\\n=== {fusion} (mean ± std over {len(SEEDS)} seeds) ===")
    for k, (m, s) in agg.items():
        print(f"  {k:32s} {m:.4f} ± {s:.4f}")

with open(os.path.join(CKPT_DIR, "results_all_fusions.json"), "w") as fp:
    json.dump(RESULTS, fp, indent=2)

table = pd.DataFrame({f: {k: f"{m:.4f}±{s:.4f}" for k, (m, s) in d["mean_std"].items()}
                      for f, d in RESULTS.items()}).T
table  # main comparison table for the paper

## Step 7b — Paired significance testing (NEW — addresses R1/R2)
Since every fusion operator is evaluated on the **same 10 seeds and the same data**, differences between
operators can be tested with a *paired* test rather than treating each operator's mean±SD independently.
Reports, for every pair of operators: mean paired difference, per-seed win counts (directly answers "does
concat actually beat x-attn on most seeds?"), a paired t-test p-value, and a Wilcoxon signed-rank p-value
(robust to the non-normality you'd expect with only ~10 samples).

In [ ]:
# Cell 10b — Paired significance testing across fusion operators (NEW, R1/R2)
def paired_tests(results, metric):
    fusions = list(results.keys())
    rows = []
    for i in range(len(fusions)):
        for j in range(i + 1, len(fusions)):
            a = np.array([r[metric] for r in results[fusions[i]]["per_seed"]])
            b = np.array([r[metric] for r in results[fusions[j]]["per_seed"]])
            diff = a - b   # negative -> fusions[i] better (lower error) than fusions[j]
            try:
                w_stat, w_p = wilcoxon(a, b)
            except Exception:
                w_stat, w_p = np.nan, np.nan
            t_stat, t_p = ttest_rel(a, b)
            rows.append(dict(
                a=fusions[i], b=fusions[j], metric=metric,
                mean_a=float(a.mean()), mean_b=float(b.mean()), mean_diff_a_minus_b=float(diff.mean()),
                a_wins=int((diff < 0).sum()), b_wins=int((diff > 0).sum()), ties=int((diff == 0).sum()),
                wilcoxon_p=float(w_p), paired_t_p=float(t_p)))
    return pd.DataFrame(rows)

sig_rmse = paired_tests(RESULTS, "rmse_log")
sig_mape = paired_tests(RESULTS, "mape")
sig_rho  = paired_tests(RESULTS, "spearman_absresid_vs_nimg")
sig_dnoimg = paired_tests(RESULTS, "delta_no_images")

print("=== RMSElog: paired comparison across fusion operators ===")
print(sig_rmse.to_string(index=False))
print("\\n=== MAPE: paired comparison ===")
print(sig_mape.to_string(index=False))
print("\\n=== Bias-rho (residual~n_img): paired comparison ===")
print(sig_rho.to_string(index=False))
print("\\n=== Missing-image degradation (delta_no_images): paired comparison ===")
print(sig_dnoimg.to_string(index=False))

for name, tbl in [("rmse", sig_rmse), ("mape", sig_mape), ("rho", sig_rho), ("dnoimg", sig_dnoimg)]:
    tbl.to_csv(os.path.join(CKPT_DIR, f"paired_significance_{name}.csv"), index=False)

print("\\nRule of thumb for the paper: report a comparison as significant only where BOTH "
      "wilcoxon_p and paired_t_p are < 0.05 AND the win count is lopsided (e.g. >= 8/10 seeds one way). "
      "Where win counts split near 50/50, state explicitly that the difference is not distinguishable "
      "from seed noise -- this directly resolves the R1/R2 critique.")

## Step 7c — Gate-stability ablation (NEW — addresses R3)
Tests Appendix A's own proposed remedies (warm-up, temperature annealing, entropy regularization)
empirically instead of leaving them as a hypothesis. Trains `gated` under all four `GATE_STABILITY`
settings across the same seeds and compares the **seed-to-seed R² / RMSElog standard deviation** —
the direct measure of the instability reviewers flagged (R² swinging 0.39–0.70 across seeds in v2).

In [ ]:
# Cell 10c — Gate-stability ablation: does warm-up / annealing / entropy fix the R^2 instability? (NEW, R3)
STAB_SETTINGS = ["none", "warmup", "anneal", "entropy"]
STAB_RESULTS = {}
for stability in STAB_SETTINGS:
    per_seed = []
    for seed in SEEDS:
        r, _m = train_one("gated", seed, verbose=False, gate_stability=stability)
        per_seed.append(r)
    agg = {k: (float(np.mean([r[k] for r in per_seed])),
               float(np.std([r[k] for r in per_seed], ddof=1))) for k in per_seed[0]}
    STAB_RESULTS[stability] = {"per_seed": per_seed, "mean_std": agg}
    print(f"\\n=== gated, stability={stability} (mean ± std over {len(SEEDS)} seeds) ===")
    for k in ["r2", "rmse_log", "mape", "spearman_absresid_vs_nimg"]:
        m, s = agg[k]
        print(f"  {k:32s} {m:.4f} ± {s:.4f}")

with open(os.path.join(CKPT_DIR, "gate_stability_ablation.json"), "w") as fp:
    json.dump(STAB_RESULTS, fp, indent=2)

stab_table = pd.DataFrame({
    s: {k: f"{m:.4f}±{sd:.4f}" for k, (m, sd) in d["mean_std"].items()}
    for s, d in STAB_RESULTS.items()
}).T
print("\\n=== Summary: does R^2 variance shrink under any stability remedy? ===")
r2_std_by_setting = {s: STAB_RESULTS[s]["mean_std"]["r2"][1] for s in STAB_SETTINGS}
for s, sd in sorted(r2_std_by_setting.items(), key=lambda kv: kv[1]):
    print(f"  {s:10s}  R^2 std = {sd:.4f}")
stab_table  # paper-ready table: gate-stability ablation

## Step 8 — Per-sample modality weights + which-photo attention (unchanged, uses winning `gated` config)
If Cell 10c shows a stability setting clearly reduces R² variance, set `GATE_STABILITY` in Cell 2 to that
value and re-run Cell 10 before continuing, so the rest of the pipeline (attribution, bias figure) uses the
stabilized model.

In [ ]:
# Cell 11 — Per-sample modality weights + which-photo attention (gated model)
_, gated_model = train_one("gated", SEEDS[0], verbose=False)
gated_model.eval()
dl_te, keys = make_loader(TENSORS, te, False)
mod_ws, img_ats = [], []
with torch.no_grad():
    for batch in dl_te:
        b = {k: v.to(DEVICE) for k, v in zip(keys, batch)}; b.pop("y")
        _, _, aux = gated_model(b, gate_temperature=1.0)
        mod_ws.append(aux["mod_weights"].cpu()); img_ats.append(aux["img_attn"].cpu())
mod_ws = torch.cat(mod_ws).numpy()   # [n_test, 3] tabular/image/geo
img_ats = torch.cat(img_ats).numpy() # [n_test, MAX_IMG]

mod_names = ["tabular", "image", "geo"]
print("mean modality weight on test set:",
      dict(zip(mod_names, mod_ws.mean(0).round(3))))

few = META["n_img"][te] <= 2; many = META["n_img"][te] >= 8
print(f"image-modality weight | few photos (<=2): {mod_ws[few,1].mean():.3f}"
      f" | many photos (>=8): {mod_ws[many,1].mean():.3f}")

## Step 9 — Bias figure (REVISED: annotates per-bin sample size, minor concern)

In [ ]:
# Cell 12 — Bias figure: |residual| vs number of live images (REVISED: bin counts annotated)
import matplotlib.pyplot as plt

_, pred, yv = run_epoch(gated_model, dl_te, keys, gate_temperature=1.0)
resid = np.abs(pred - yv)
nimg_te = META["n_img"][te]

bins = pd.DataFrame({"n_img": nimg_te, "abs_resid": resid}) \
         .groupby("n_img")["abs_resid"].agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(bins.index, bins["mean"])
for rect, n in zip(bars, bins["count"]):
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height(), f"n={n}",
            ha="center", va="bottom", fontsize=8)
ax.set_xlabel("live images per property"); ax.set_ylabel("mean |log-price residual|")
ax.set_title("Modality-imbalance bias check (flat = unbiased); n annotated per bin")
plt.tight_layout(); plt.savefig(os.path.join(CKPT_DIR, "bias_nimages.png"), dpi=150)
plt.show()
print(bins)

## Step 10 (optional) — LLM structured text features (unchanged from v2)
`mod_meta` is now 2-D `[richness, quality]`. If you add text features, extend `mod_meta` to 3 columns and
set `d_meta=3` everywhere in Cell 7 (`GMU(..., d_meta)`, `late`'s `self.w`).

In [ ]:
# Cell 13 — LLM extraction template (plug in your LLM call) (unchanged from v2)
SCHEMA = {
    "has_kitchen_mention": "Y/N",
    "kitchen_renovated": "Y/N/Unknown",
    "overall_condition": "1-5 ordinal (1=needs work, 5=immaculate)",
    "recently_renovated": "Y/N/Unknown",
    "view_quality": "0-3 (0=none, 3=premium water/city view)",
    "outdoor_space": "Y/N",
    "negations": "list of amenities explicitly stated ABSENT (e.g. 'no parking')",
    "marketing_hype": "1-5 promotional tone",
}
PROMPT = ("You extract structured facts from a real-estate listing description.\n"
          "Answer ONLY with a JSON object matching this schema (no prose):\n{schema}\n"
          "Rules: 'Unknown' when unstated; be conservative; catch negations.\n"
          'Description:\n"""{description}"""')

def ask_llm(prompt: str) -> str:
    raise NotImplementedError("plug in Anthropic API / local LLM here")

def extract_text_features(desc_csv, text_col="description", id_col="proID"):
    ddf = pd.read_csv(desc_csv)
    out = []
    for _, r in ddf.iterrows():
        text = str(r[text_col])[:4000]
        try:
            ans = json.loads(ask_llm(PROMPT.format(
                schema=json.dumps(SCHEMA), description=text)))
        except Exception:
            ans = {}
        ans[id_col] = r[id_col]; ans["desc_word_count"] = len(text.split())
        out.append(ans)
    feats = pd.DataFrame(out)
    for c in ["has_kitchen_mention", "kitchen_renovated",
              "recently_renovated", "outdoor_space"]:
        if c in feats: feats[c] = feats[c].map({"Y": 1, "N": 0}).fillna(-1)
    feats.to_csv("llm_text_features.csv", index=False)
    return feats

# feats = extract_text_features("2_-_Description.csv")
# then: merge into x_num (Cell 8), set txt_len = word counts, rebuild TENSORS, rerun Cell 10

## Step 11 — Second-backbone / second-dataset robustness runs (NEW — addresses R7)
Two independent generality checks the reviewers asked for. Neither requires new code beyond what's above
— only re-pointing config and re-running.

**(a) Second visual backbone.** In Cell 2, set:
```python
BACKBONE_NAME = "vit_base_patch16_clip_224.openai"
```
then re-run Cells 4 → 10. Embeddings are namespaced by `BACKBONE_TAG`, so both runs' checkpoints coexist
in `CKPT_DIR` and `RESULTS` from each run can be saved under different filenames
(`results_all_fusions_{BACKBONE_TAG}.json`) for a side-by-side table in the paper.

**(b) Second dataset.** Point `PROP_CSV` / `PIC_CSV` at a second corpus with the same schema (`ID`, `price`,
`sold_date`, `Lat`, `Lng`, `proType`, `agency_name`, `Locality`, `Postal Code`, amenity columns; and a
`proID`/`picNo`/`picAddr` picture table). Use a fresh `CKPT_DIR` (e.g. append a city suffix) so the two
runs don't overwrite each other, then run Cells 3 → 10c end to end. If no second public AVM dataset with
photo URLs is available, the smallest defensible substitute is a **held-out geographic split** of the same
corpus (e.g. train on inner-Melbourne suburbs, test on outer-Melbourne) as a partial generality check —
code below does this split as a drop-in alternative to the temporal split in Cell 8.

In [ ]:
# Cell 14 — Optional partial-generality check: geographic (rather than temporal) holdout (NEW, R7)
# Use this ONLY as a supplementary robustness check -- the primary protocol stays the strict temporal
# split (Cell 8). This trains/evaluates the same models on a spatial holdout to see whether performance
# and the bias-rho finding transfer across neighborhoods, as a partial substitute for a second dataset.
GEO_HOLDOUT = False   # flip to True to run this cell's split instead of the temporal one

if GEO_HOLDOUT:
    lat_median = np.median(latlng[tr, 0])
    geo_tr = np.where(latlng[:, 0] <= lat_median)[0]
    geo_te = np.where(latlng[:, 0] >  lat_median)[0]
    rng = np.random.RandomState(0)
    rng.shuffle(geo_tr)
    n_va = int(0.125 * len(geo_tr))   # ~matches original 10% val share of the 80% non-test portion
    geo_va, geo_tr2 = geo_tr[:n_va], geo_tr[n_va:]
    print(f"geo split: train {len(geo_tr2)} | val {len(geo_va)} | test {len(geo_te)}")
    print("Re-point tr, va, te to geo_tr2, geo_va, geo_te and re-run Cell 10 to get the geo-holdout table.")
    tr, va, te = geo_tr2, geo_va, geo_te
else:
    print("GEO_HOLDOUT is False -- primary temporal split (Cell 8) remains active. "
          "Set True and re-run this cell, then re-run Cell 10, for the supplementary robustness table.")

## Step 11c — Genuine second dataset (NEW — strengthens R7 beyond the geo-holdout)

Cell 14's geographic holdout is a same-corpus partial check. This section adds an **actual second,
independent dataset**: Ahmed & Moustafa (2016), *"House price estimation from visual and textual
features"* (github.com/emanhamed/Houses-dataset) — 535 US properties (mixed CA/AZ zip codes), 4 photos
each (bedroom/bathroom/kitchen/frontal), price + bed/bath/sqft/zipcode metadata.

**Everything above (Cells 1-10c, encoders, fusion modules, training loop, confound check, significance
tests, gate-stability ablation) is unchanged.** This section only prepares a second corpus in the same
schema and re-points config, exactly as Cell 32's note (b) anticipated.

**Read before trusting these numbers — four honest caveats:**
1. **No real `sold_date`.** The source has no sale date. We assign a synthetic sequential pseudo-date so
   the temporal-split code path runs, but this is **not** a genuine chronology — don't use this dataset
   to claim leak-free temporal generalization. It's a structural check only: does the architecture and
   bias story replicate on a different city/backbone-input distribution at all?
2. **No real `Lat`/`Lng`.** We derive a deterministic pseudo-coordinate per zip code (hash-based jitter
   around a fixed anchor), not real geocoding (no geocoding API is reachable from this environment).
   Good enough to give the KNN comps feature and the eta² locality check something structurally valid;
   not to be plotted as a real map.
3. **No real `agency_name`.** Rather than fabricate one, we alias it to a zip-code bucket and flag this,
   so `eta²(agency)` here is really `eta²(zipcode-bucket)`, not independent information.
4. **Fixed 4 photos/property — no organic missingness.** To exercise richness/gating at all (the entire
   point of this check), we **inject synthetic missingness**: photos are dropped more often for cheaper
   properties (mirrors the reviewer's "budget agencies show fewer photos" worry). This is a *known,
   controlled* confound — which doubles as a sanity check on Cell 5b's confound-detection code, since we
   can compare its estimated price-correlation against the true injected one (printed at the end of the
   next cell).

Run once:
```bash
git clone --depth 1 https://github.com/emanhamed/Houses-dataset.git
```
then run the next two cells.

In [ ]:
# Cell 11c-1 — Build the second dataset in the pipeline's schema (NEW, R7)
import os, hashlib, shutil
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SRC_DIR2 = "Houses-dataset/Houses Dataset"     # from the git clone above
OUT_DIR2 = "second_dataset"
IMG_OUT2 = os.path.join(OUT_DIR2, "images2")
os.makedirs(IMG_OUT2, exist_ok=True)
RNG2 = np.random.RandomState(0)

info = pd.read_csv(os.path.join(SRC_DIR2, "HousesInfo.txt"), sep=" ", header=None,
                    names=["bed", "bath", "sqft", "zipcode", "price"])
info["original_id"] = np.arange(1, len(info) + 1)   # matches repo's 1..535 image filenames
n2 = len(info)
info = info.sample(frac=1.0, random_state=0).reset_index(drop=True)
info["ID"] = np.arange(1, n2 + 1)

ANCHOR_LAT, ANCHOR_LNG = 34.0, -117.5   # pseudo-geocoding anchor -- NOT real geocoding, see caveat 2

def pseudo_latlng(zipcode):
    h = int(hashlib.md5(str(zipcode).encode()).hexdigest(), 16)
    dlat = ((h % 10000) / 10000 - 0.5) * 4.0
    dlng = (((h // 10000) % 10000) / 10000 - 0.5) * 4.0
    return ANCHOR_LAT + dlat, ANCHOR_LNG + dlng

ll = info["zipcode"].apply(pseudo_latlng)
info["Lat"], info["Lng"] = ll.apply(lambda t: t[0]), ll.apply(lambda t: t[1])
info["sold_date"] = pd.date_range("2018-01-01", periods=n2, freq="D").strftime("%Y-%m-%d")  # caveat 1
info["proType"] = "SFH"
info["Locality"] = "zip_" + info["zipcode"].astype(str)
info["Postal Code"] = info["zipcode"].astype(str)
info["agency_name"] = "zipband_" + (info["zipcode"] % 7).astype(str)   # caveat 3

property_cols = ["ID", "price", "sold_date", "Lat", "Lng", "proType", "agency_name",
                  "Locality", "Postal Code", "bed", "bath", "sqft"]
props_out = info[property_cols].copy()
props_out.to_csv(os.path.join(OUT_DIR2, "Property_full_Table2.csv"), index=False)

# picture table with INJECTED richness variability (caveat 4)
kinds = ["frontal", "bedroom", "bathroom", "kitchen"]
price_tercile = pd.qcut(info["price"], 3, labels=["low", "mid", "high"])
keep_prob_map = {"low": 0.55, "mid": 0.75, "high": 0.92}

manifest_rows = []
for _, row in info.iterrows():
    pid, oid = int(row["ID"]), int(row["original_id"])
    kp = keep_prob_map[price_tercile.loc[row.name]]
    for pic_no, kind in enumerate(kinds):
        src_path = os.path.join(SRC_DIR2, f"{oid}_{kind}.jpg")
        if os.path.exists(src_path) and RNG2.rand() < kp:
            dst_name = f"{pid}_{pic_no}.jpg"
            shutil.copyfile(src_path, os.path.join(IMG_OUT2, dst_name))
            manifest_rows.append((pid, pic_no, dst_name))

pic_manifest = pd.DataFrame(manifest_rows, columns=["proID", "picNo", "filename"])
pic_addr = pic_manifest.copy()
pic_addr["picAddr"] = pic_addr["filename"].apply(lambda f: os.path.join(IMG_OUT2, f))
pic_addr[["proID", "picNo", "picAddr"]].to_csv(os.path.join(OUT_DIR2, "Picture2.csv"), index=False)
pic_manifest.to_csv(os.path.join(OUT_DIR2, "image_manifest2.csv"), index=False)

n_img_per_prop = pic_manifest.groupby("proID").size().reindex(info["ID"], fill_value=0)
rho_true, p_true = spearmanr(n_img_per_prop.values,
                              info.set_index("ID").loc[n_img_per_prop.index, "price"].values)
print(f"second dataset: {n2} properties, {len(pic_manifest)}/{4*n2} photos kept "
      f"({100*len(pic_manifest)/(4*n2):.1f}%)")
print(f"TRUE injected spearman(n_img, price) = {rho_true:.4f} (p={p_true:.2e}) "
      "-- compare this to what Cell 5b measures below, as a sanity check on the confound-detection code.")

In [ ]:
# Cell 11c-2 — Re-point config to the second dataset and re-run Cells 4->5->5b->8->9->10->10b->10c
# (Cell 3's live downloader is skipped -- images are already local; image_manifest2.csv stands in
# for what Cell 3 would have produced.)
PROP_CSV = "second_dataset/Property_full_Table2.csv"
PIC_CSV  = "second_dataset/Picture2.csv"
IMG_DIR  = "second_dataset/images2"
MAX_IMG  = 4                                   # this dataset's max is 4 photos, not 10
CKPT_DIR = "ckpt_second_dataset"               # separate checkpoint dir -- doesn't clobber primary run
os.makedirs(CKPT_DIR, exist_ok=True)
shutil.copyfile("second_dataset/image_manifest2.csv", os.path.join(CKPT_DIR, "image_manifest.csv"))

print("Re-pointed to the second dataset. Now run, in order: Cell 4 (embeddings) -> Cell 5 (tensors) ->")
print("Cell 5b (confound check -- compare its rho against the TRUE injected rho printed above) ->")
print("Cell 8 (feature build/split) -> Cell 9 (train helpers, if not already defined) -> Cell 10 (full")
print("comparison) -> Cell 10b (paired significance) -> Cell 10c (gate-stability ablation).")
print()
print("When done, save RESULTS separately (e.g. json.dump(RESULTS, open('results_second_dataset.json','w')))")
print("before re-running Cell 2 to restore PROP_CSV/PIC_CSV/MAX_IMG/CKPT_DIR to the primary dataset.")
print()
print("NOTE: this smoke-tested end-to-end with pretrained=False (no Hub access in the sandbox used to")
print("build this cell) -- code path and schema are verified, but you must re-run with the real")
print("pretrained backbone (pretrained=True, as in Cell 4) for numbers that mean anything.")

## Evaluation protocol summary (REVISED, paper-ready) + LaTeX table exporter

In [ ]:
# Cell 15 — LaTeX table exporter (NEW — practical addition for the revised manuscript)
def to_latex_table(results, metrics_to_show=("rmse_log", "mape", "delta_no_images",
                                              "spearman_absresid_vs_nimg"), caption="", label=""):
    rows = []
    for fusion, d in results.items():
        m = d["mean_std"]
        row = [fusion] + [f"{m[k][0]:.3f} $\\pm$ {m[k][1]:.3f}" for k in metrics_to_show]
        rows.append(row)
    header = ["Fusion operator"] + [k.replace("_", "\\_") for k in metrics_to_show]
    col_spec = "l" + "c" * len(metrics_to_show)
    lines = [
        "\\begin{table}[t]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        f"\\begin{{tabular}}{{{col_spec}}}",
        "\\toprule",
        " & ".join(header) + " \\\\",
        "\\midrule",
    ]
    for row in rows:
        lines.append(" & ".join(row) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\\n".join(lines)

latex_main = to_latex_table(RESULTS, caption="Fusion operator comparison, mean $\\pm$ std over "
                             f"{len(SEEDS)} seeds.", label="tab:fusion_comparison")
latex_stab = to_latex_table(STAB_RESULTS, metrics_to_show=("r2", "rmse_log", "mape"),
                             caption="Gate-stability ablation: warm-up/annealing/entropy vs. baseline.",
                             label="tab:gate_stability")

with open(os.path.join(CKPT_DIR, "table_fusion_comparison.tex"), "w") as f:
    f.write(latex_main)
with open(os.path.join(CKPT_DIR, "table_gate_stability.tex"), "w") as f:
    f.write(latex_stab)

print(latex_main)
print()
print(latex_stab)

In [ ]:
# Cell 16 — Evaluation protocol summary (REVISED, paper-ready)
print("""
Evaluation protocol (v3):
- Target: log(price); temporal 70/10/20 split by sold_date (no market-trend leakage)
- Metrics: RMSE/MAE on log-price, MAPE & R^2 on price; mean +/- std over {n_seeds} seeds (was 3 in v2)
- Significance: paired Wilcoxon + paired t-test across shared seeds for every operator pair (Cell 10b)
- Confounding check: n_img regressed/associated against agency, locality, postal, price, sale period (Cell 5b)
- Quality signal: mod_meta = [richness (live-photo count/MAX_IMG), quality (mean log-Laplacian-variance
  sharpness of live photos)] -- resolves the "quality-aware = count only" critique (R6)
- Capacity control: gated_wide, parameter-matched to xattn within ~1%, isolates mechanism from capacity (R5)
- Stability: gate temperature warm-up/annealing + entropy regularization tested against baseline gate,
  compared on seed-to-seed R^2/RMSE standard deviation (Cell 10c)
- Generality: backbone-swap hook (BACKBONE_NAME) + geographic-holdout supplementary split (Cell 14)
- Evidence per research question:
    RQ1 (fusion > concat?)      -> Cell 10 table + Cell 10b paired significance
    RQ2 (imbalance bias real &
         fixable, and is it
         confounded?)            -> Cell 5b confound check + Cell 12 bias figure + Cell 10b rho comparison
    RQ3 (per-prediction
         attribution)            -> Cell 11 (gate weights + image attention)
    (new) gate instability        -> Cell 10c stability ablation
    (new) generality               -> Cell 14 geo-holdout / BACKBONE_NAME swap
""".format(n_seeds=len(SEEDS)))

print("Resume after a Colab disconnect: rerun Cells 2 -> 4 -> 5 -> 5b -> 8 (fast once checkpoints exist "
      "on Drive; Cell 3 only if images aren't downloaded), then continue from wherever you were.")